In [1]:
import os
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import softmax
from itertools import product

import utilities as utils

In [2]:
save_data_flag = True

## Bead prediction experiment

In [5]:
beads_datadir = "./Data/beads/"
with open(beads_datadir + 'flipped_sjs.txt', 'r',  encoding='utf-16') as f:
    flipped_sjs = [int(line.strip()) for line in f.readlines()]
beads_trial_seq = pd.read_csv(beads_datadir + "trial_sequence.csv")
preprocdf = pd.read_csv(beads_datadir + "sj-preproc-data.csv")

wsize = 7 # number of observed beads to include in the "window" used to compute choice probabilities and posterior probabilities for each unique X (sequence of observed beads of length wsize, from current trial backwards)

# generative parameters for the bead-prediction experiment
h_=0.99 # jar stay probability
p0_=0.8 # probability of drawing bead type 0 from jar 0
p1_=0.2 # probability of drawing bead type 0 from jar 1
P0 = np.array([[0.5,0.5]]) # prior over jars for computing posterior probabilities over hidden markov process
H = np.ones((2,2)) - np.abs(np.eye(2)*-1 + h_) # transition matrix
E = np.vstack((np.array([[1,0]])*p0_ + np.array([[0,1]])*(1-p0_),np.array([[1,0]])*p1_ + np.array([[0,1]])*(1-p1_))) # emission matrix

modeldf = pd.DataFrame({
    'subject_ID': [],
    'subject_index': [],
    'run': [],
    'opt_betastar': [],
    'opt_loss': [],
    'opt_BIC': [],
    'opt_rel_entropy': [],
    'oneback_betastar': [],
    'oneback_loss': [],
    'oneback_BIC': [],
    'oneback_rel_entropy': [],
    'fullparam_betastar': [],
    'fullparam_h': [],
    'fullparam_loss': [],
    'fullparam_BIC': [],
    'fullparam_rel_entropy': [],
    'best_model': [],
    'best_model_betastar': [],
    'best_model_loss': [],
    'best_model_BIC': [],
    'best_model_rel_entropy': []
})

beads_emp, jars_emp = utils.getXY_beads(beads_trial_seq['bead'].to_numpy(), beads_trial_seq['jar'].to_numpy(), wsize)
beads_emp_enc, _ = utils.getXY_beads(beads_trial_seq['bead'].to_numpy(), beads_trial_seq['jar'].to_numpy(), wsize,encodeX=True)
beads_emp1b, jars_emp1b = utils.getXY_beads(beads_trial_seq['bead'].to_numpy(), beads_trial_seq['jar'].to_numpy(), wsize=1,ref_wsize=wsize)
p_YgX_opt = utils.P_jar_g_beads(beads_emp,E,H,P0)
p_YgX_1b = utils.P_jar_g_beads(beads_emp1b,E,H,P0)

Ntrials = beads_emp.shape[0]

fully_opt_model = lambda params: softmax(params[0]*p_YgX_opt, axis=1)
oneback_model = lambda params: softmax(params[0]*p_YgX_1b, axis=1)

def fullparam_model(params):
    betastar, h = params
    H_ = np.ones((2,2)) - np.abs(np.eye(2)*-1 + h) # transition matrix
    p_YgX_fullparam = utils.P_jar_g_beads(beads_emp,E,H_,P0)
    return softmax(betastar*p_YgX_fullparam, axis=1)

def fit_fullparam_model(obj_fn):
    bounds = [(0.0, None),(0.0, 1.0)]
    best_result = None
    best_loss = np.inf
    betastar0 = np.array([0.1, 1., 3., 5., 8.])
    h0 = np.array([0.6, 0.7, 0.8, 0.9, 0.99])
    inds = product(range(len(betastar0)),range(len(h0)))
    for ind in inds:
        X0 = np.array([betastar0[ind[0]], h0[ind[1]]])
        result = minimize(obj_fn,X0,bounds=bounds,options={'eps': 1e-19})
        if result.fun <= best_loss:
            best_result = result
            best_loss = result.fun
    if best_result is None:
        return result
    else:
        return best_result

for sj in preprocdf['subject_index'].unique():
    for run in [1,2]:
        df_ = preprocdf[(preprocdf['subject_index']==sj) & (preprocdf['run']==run)]

        sjdf = pd.DataFrame({
            'subject_ID': [df_['subject_ID'].iloc[0]],
            'subject_index': [sj],
            'run': [run],
        })

        choices = utils.getR_beads(df_['choice'].to_numpy(),wsize)
        if sj in flipped_sjs:
            choices = 1 - choices
        sj_choice_probs = utils.get_empirical_pRgX(beads_emp_enc,choices)

        oneback_obj = lambda params: utils.compute_neg_sum_log_likelihood(choices,oneback_model,params)
        result = minimize(oneback_obj,np.array([1.]),bounds=[(0.0, None)])
        sjdf['oneback_betastar'] = result.x[0]
        sjdf['oneback_loss'] = result.fun
        sjdf['oneback_BIC'] = utils.compute_BIC(1, Ntrials, result.fun) # 1 free parameter (betastar) for this model
        sjdf['oneback_rel_entropy'] = utils.compute_rel_entropy(sj_choice_probs, oneback_model(result.x))
        sjdf['best_model'] = 'oneback'
        sjdf['best_model_betastar'] = result.x[0]
        sjdf['best_model_loss'] = result.fun
        sjdf['best_model_BIC'] = sjdf['oneback_BIC'].iloc[0]
        sjdf['best_model_rel_entropy'] = sjdf['oneback_rel_entropy'].iloc[0]

        fully_opt_obj = lambda params: utils.compute_neg_sum_log_likelihood(choices,fully_opt_model,params)
        result = minimize(fully_opt_obj,np.array([1.]),bounds=[(0.0, None)])
        sjdf['opt_betastar'] = result.x[0]
        sjdf['opt_loss'] = result.fun
        sjdf['opt_BIC'] = utils.compute_BIC(1, Ntrials, result.fun) # 1 free parameter (betastar) for this model
        sjdf['opt_rel_entropy'] = utils.compute_rel_entropy(sj_choice_probs, fully_opt_model(result.x))
        if sjdf['opt_BIC'].iloc[0] < sjdf['best_model_BIC'].iloc[0]:
            sjdf['best_model'] = 'fully_optimal'
            sjdf['best_model_betastar'] = result.x[0]
            sjdf['best_model_loss'] = result.fun
            sjdf['best_model_BIC'] = sjdf['opt_BIC'].iloc[0]
            sjdf['best_model_rel_entropy'] = sjdf['opt_rel_entropy'].iloc[0]

        fullparam_obj = lambda params: utils.compute_neg_sum_log_likelihood(choices,fullparam_model,params)
        result = fit_fullparam_model(fullparam_obj)
        sjdf['fullparam_betastar'] = result.x[0]
        sjdf['fullparam_h'] = result.x[1]
        sjdf['fullparam_loss'] = result.fun
        sjdf['fullparam_BIC'] = utils.compute_BIC(2, Ntrials, result.fun) # 2 free parameters (betastar, h) for this model
        sjdf['fullparam_rel_entropy'] = utils.compute_rel_entropy(sj_choice_probs, fullparam_model(result.x))
        if sjdf['fullparam_BIC'].iloc[0] < sjdf['best_model_BIC'].iloc[0]:
            sjdf['best_model'] = 'fullparam'
            sjdf['best_model_betastar'] = result.x[0]
            sjdf['best_model_loss'] = result.fun
            sjdf['best_model_BIC'] = sjdf['fullparam_BIC'].iloc[0]
            sjdf['best_model_rel_entropy'] = sjdf['fullparam_rel_entropy'].iloc[0]

        modeldf = pd.concat((modeldf, sjdf), ignore_index=True)

    # print(f'Subject {sj} done')

if save_data_flag:
    modeldf.to_csv(beads_datadir + 'model_fits.csv', index=False)


/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: divide by zero encountered in log
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: invalid value encountered in multiply
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: divide by zero encountered in log
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: invalid value encountered in multiply
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: divide by zero encountered in log
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_pr

## Horse prediction experiments

In [4]:
horses_datadir = "./Data/horses/"

exps1 = ['lowWS','midWS','highWS']
exps2 = ['learning','reward','speed_accuracy']
conds_dict = {
    'learning': ['run_1', 'run_2'],
    'reward': ['low_reward', 'high_reward'],
    'speed_accuracy': ['fast', 'slow']
}

preproc_dict = {
    'lowWS': pd.read_csv(horses_datadir + "lowWS/sj-preproc-data.csv"),
    'midWS': pd.read_csv(horses_datadir + "midWS_learning/sj-preproc-data.csv"),
    'highWS': pd.read_csv(horses_datadir + "highWS/sj-preproc-data.csv"),
    'learning': pd.read_csv(horses_datadir + "midWS_learning/sj-preproc-data.csv"),
    'reward': pd.read_csv(horses_datadir + "reward/sj-preproc-data.csv"),
    'speed_accuracy': pd.read_csv(horses_datadir + "speed_accuracy/sj-preproc-data.csv")
}

paramdict = {
    'lowWS': {
        'weakLLR': 0.45,
        'WSratio': 1.3,
        'p1': 0.06
    },
    'midWS': {
        'weakLLR': 0.2,
        'WSratio': 2.5,
        'p1': 0.08
    },
    'highWS': {
        'weakLLR': 0.18,
        'WSratio': 6.3,
        'p1': 0.02
    }
}

p_Y = np.array([[0.5, 0.5]])

def fullparam_model_horses(params,shapes_emp,exp):
    betastar, WSratio = params
    p_YgX_fullparam = utils.P_horse_g_shapecomb(shapes_emp,paramdict[exp]['weakLLR'],WSratio,paramdict[exp]['p1'],p_Y)
    return softmax(betastar*p_YgX_fullparam, axis=1)

modeldf_dict = {}

for exp in exps1:
    modeldf_dict[exp] = pd.DataFrame({
        'subject_ID': [],
        'subject_index': [],
        'opt_betastar': [],
        'opt_loss': [],
        'opt_BIC': [],
        'opt_rel_entropy': [],
        'equalweights_betastar': [],
        'equalweights_loss': [],
        'equalweights_BIC': [],
        'equalweights_rel_entropy': [],
        'ignoreweak_betastar': [],
        'ignoreweak_loss': [],
        'ignoreweak_BIC': [],
        'ignoreweak_rel_entropy': [],
        'fullparam_betastar': [],
        'fullparam_WSratio': [],
        'fullparam_loss': [],
        'fullparam_BIC': [],
        'fullparam_rel_entropy': [],
        'best_model': [],
        'best_model_betastar': [],
        'best_model_loss': [],
        'best_model_BIC': [],
        'best_model_rel_entropy': []
    })

    for sj in preproc_dict[exp]['subject_index'].unique():
        df_ = preproc_dict[exp][preproc_dict[exp]['subject_index']==sj]

        sjdf = pd.DataFrame({
            'subject_ID': [df_['subject_ID'].iloc[0]],
            'subject_index': [sj],
        })

        shapes_emp_ = utils.split_to_four_digits(df_['observation_encoding'].to_numpy())
        horses_emp_ = df_['latent_state'].to_numpy()
        choices = df_['choice'].to_numpy()
        sj_choice_probs = utils.get_empirical_pRgX(df_['observation_encoding'].to_numpy(),choices)

        Ntrials = choices.shape[0]

        p_YgX_ew = utils.P_horse_g_shapecomb(shapes_emp_,paramdict[exp]['weakLLR'],paramdict[exp]['WSratio'],paramdict[exp]['p1'],p_Y,utils.P_shapecomb_g_horse_ew)
        equalweights_model = lambda params: softmax(params[0]*p_YgX_ew, axis=1)
        equalweights_obj = lambda params: utils.compute_neg_sum_log_likelihood(choices,equalweights_model,params)
        result = minimize(equalweights_obj,np.array([1.]),bounds=[(0.0, None)])
        sjdf['equalweights_betastar'] = result.x[0]
        sjdf['equalweights_loss'] = result.fun
        sjdf['equalweights_BIC'] = utils.compute_BIC(1, Ntrials, result.fun)
        sjdf['equalweights_rel_entropy'] = utils.compute_rel_entropy(sj_choice_probs, equalweights_model(result.x))
        sjdf['best_model'] = 'equalweights'
        sjdf['best_model_betastar'] = result.x[0]
        sjdf['best_model_loss'] = result.fun
        sjdf['best_model_BIC'] = sjdf['equalweights_BIC'].iloc[0]
        sjdf['best_model_rel_entropy'] = sjdf['equalweights_rel_entropy'].iloc[0]

        p_YgX_iw = utils.P_horse_g_shapecomb(shapes_emp_,paramdict[exp]['weakLLR'],paramdict[exp]['WSratio'],paramdict[exp]['p1'],p_Y,utils.P_shapecomb_g_horse_iw)
        ignoreweak_model = lambda params: softmax(params[0]*p_YgX_iw, axis=1)
        ignoreweak_obj = lambda params: utils.compute_neg_sum_log_likelihood(choices,ignoreweak_model,params)
        result = minimize(ignoreweak_obj,np.array([1.]),bounds=[(0.0, None)])
        sjdf['ignoreweak_betastar'] = result.x[0]
        sjdf['ignoreweak_loss'] = result.fun
        sjdf['ignoreweak_BIC'] = utils.compute_BIC(1, Ntrials, result.fun)
        sjdf['ignoreweak_rel_entropy'] = utils.compute_rel_entropy(sj_choice_probs, ignoreweak_model(result.x))
        if sjdf['ignoreweak_BIC'].iloc[0] < sjdf['best_model_BIC'].iloc[0]:
            sjdf['best_model'] = 'ignoreweak'
            sjdf['best_model_betastar'] = result.x[0]
            sjdf['best_model_loss'] = result.fun
            sjdf['best_model_BIC'] = sjdf['ignoreweak_BIC'].iloc[0]
            sjdf['best_model_rel_entropy'] = sjdf['ignoreweak_rel_entropy'].iloc[0]

        p_YgX_opt = utils.P_horse_g_shapecomb(shapes_emp_,paramdict[exp]['weakLLR'],paramdict[exp]['WSratio'],paramdict[exp]['p1'],p_Y)
        fully_opt_model = lambda params: softmax(params[0]*p_YgX_opt, axis=1)
        fully_opt_obj = lambda params: utils.compute_neg_sum_log_likelihood(choices,fully_opt_model,params)
        result = minimize(fully_opt_obj,np.array([1.]),bounds=[(0.0, None)])
        sjdf['opt_betastar'] = result.x[0]
        sjdf['opt_loss'] = result.fun
        sjdf['opt_BIC'] = utils.compute_BIC(1, Ntrials, result.fun)
        sjdf['opt_rel_entropy'] = utils.compute_rel_entropy(sj_choice_probs, fully_opt_model(result.x))
        if sjdf['opt_BIC'].iloc[0] < sjdf['best_model_BIC'].iloc[0]:
            sjdf['best_model'] = 'fully_optimal'
            sjdf['best_model_betastar'] = result.x[0]
            sjdf['best_model_loss'] = result.fun
            sjdf['best_model_BIC'] = sjdf['opt_BIC'].iloc[0]
            sjdf['best_model_rel_entropy'] = sjdf['opt_rel_entropy'].iloc[0]

        fullparam_model = lambda params: fullparam_model_horses(params,shapes_emp_,exp)
        fullparam_obj = lambda params: utils.compute_neg_sum_log_likelihood(choices,fullparam_model,params)
        result = minimize(fullparam_obj,np.array([1., 1.]),bounds=[(0.0, None),(0.0, None)])
        sjdf['fullparam_betastar'] = result.x[0]
        sjdf['fullparam_WSratio'] = result.x[1]
        sjdf['fullparam_loss'] = result.fun
        sjdf['fullparam_BIC'] = utils.compute_BIC(2, Ntrials, result.fun)
        sjdf['fullparam_rel_entropy'] = utils.compute_rel_entropy(sj_choice_probs, fullparam_model(result.x))
        if sjdf['fullparam_BIC'].iloc[0] < sjdf['best_model_BIC'].iloc[0]:
            sjdf['best_model'] = 'fullparam'
            sjdf['best_model_betastar'] = result.x[0]
            sjdf['best_model_loss'] = result.fun
            sjdf['best_model_BIC'] = sjdf['fullparam_BIC'].iloc[0]
            sjdf['best_model_rel_entropy'] = sjdf['fullparam_rel_entropy'].iloc[0]

        modeldf_dict[exp] = pd.concat((modeldf_dict[exp], sjdf), ignore_index=True)

    print(f'Experiment {exp} done')

for exp in exps2:
    modeldf_dict[exp] = pd.DataFrame({
        'subject_ID': [],
        'subject_index': [],
        'condition': [],
        'opt_betastar': [],
        'opt_loss': [],
        'opt_BIC': [],
        'opt_rel_entropy': [],
        'equalweights_betastar': [],
        'equalweights_loss': [],
        'equalweights_BIC': [],
        'equalweights_rel_entropy': [],
        'ignoreweak_betastar': [],
        'ignoreweak_loss': [],
        'ignoreweak_BIC': [],
        'ignoreweak_rel_entropy': [],
        'fullparam_betastar': [],
        'fullparam_WSratio': [],
        'fullparam_loss': [],
        'fullparam_BIC': [],
        'fullparam_rel_entropy': [],
        'best_model': [],
        'best_model_betastar': [],
        'best_model_loss': [],
        'best_model_BIC': [],
        'best_model_rel_entropy': []
    })

    for cond in conds_dict[exp]:
        for sj in preproc_dict[exp]['subject_index'].unique():
            df_ = preproc_dict[exp][(preproc_dict[exp]['subject_index']==sj) & (preproc_dict[exp]['condition']==cond)]

            sjdf = pd.DataFrame({
                'subject_ID': [df_['subject_ID'].iloc[0]],
                'condition': [cond],
                'subject_index': [sj],
            })

            shapes_emp_ = utils.split_to_four_digits(df_['observation_encoding'].to_numpy())
            horses_emp_ = df_['latent_state'].to_numpy()
            choices = df_['choice'].to_numpy()
            sj_choice_probs = utils.get_empirical_pRgX(df_['observation_encoding'].to_numpy(),choices)

            Ntrials = choices.shape[0]

            p_YgX_ew = utils.P_horse_g_shapecomb(shapes_emp_,paramdict['midWS']['weakLLR'],paramdict['midWS']['WSratio'],paramdict['midWS']['p1'],p_Y,utils.P_shapecomb_g_horse_ew)
            equalweights_model = lambda params: softmax(params[0]*p_YgX_ew, axis=1)
            equalweights_obj = lambda params: utils.compute_neg_sum_log_likelihood(choices,equalweights_model,params)
            result = minimize(equalweights_obj,np.array([1.]),bounds=[(0.0, None)])
            sjdf['equalweights_betastar'] = result.x[0]
            sjdf['equalweights_loss'] = result.fun
            sjdf['equalweights_BIC'] = utils.compute_BIC(1, Ntrials, result.fun)
            sjdf['equalweights_rel_entropy'] = utils.compute_rel_entropy(sj_choice_probs, equalweights_model(result.x))
            sjdf['best_model'] = 'equalweights'
            sjdf['best_model_betastar'] = result.x[0]
            sjdf['best_model_loss'] = result.fun
            sjdf['best_model_BIC'] = sjdf['equalweights_BIC'].iloc[0]
            sjdf['best_model_rel_entropy'] = sjdf['equalweights_rel_entropy'].iloc[0]

            p_YgX_iw = utils.P_horse_g_shapecomb(shapes_emp_,paramdict['midWS']['weakLLR'],paramdict['midWS']['WSratio'],paramdict['midWS']['p1'],p_Y,utils.P_shapecomb_g_horse_iw)
            ignoreweak_model = lambda params: softmax(params[0]*p_YgX_iw, axis=1)
            ignoreweak_obj = lambda params: utils.compute_neg_sum_log_likelihood(choices,ignoreweak_model,params)
            result = minimize(ignoreweak_obj,np.array([1.]),bounds=[(0.0, None)])
            sjdf['ignoreweak_betastar'] = result.x[0]
            sjdf['ignoreweak_loss'] = result.fun
            sjdf['ignoreweak_BIC'] = utils.compute_BIC(1, Ntrials, result.fun)
            sjdf['ignoreweak_rel_entropy'] = utils.compute_rel_entropy(sj_choice_probs, ignoreweak_model(result.x))
            if sjdf['ignoreweak_BIC'].iloc[0] < sjdf['best_model_BIC'].iloc[0]:
                sjdf['best_model'] = 'ignoreweak'
                sjdf['best_model_betastar'] = result.x[0]
                sjdf['best_model_loss'] = result.fun
                sjdf['best_model_BIC'] = sjdf['ignoreweak_BIC'].iloc[0]
                sjdf['best_model_rel_entropy'] = sjdf['ignoreweak_rel_entropy'].iloc[0]

            p_YgX_opt = utils.P_horse_g_shapecomb(shapes_emp_,paramdict['midWS']['weakLLR'],paramdict['midWS']['WSratio'],paramdict['midWS']['p1'],p_Y)
            fully_opt_model = lambda params: softmax(params[0]*p_YgX_opt, axis=1)
            fully_opt_obj = lambda params: utils.compute_neg_sum_log_likelihood(choices,fully_opt_model,params)
            result = minimize(fully_opt_obj,np.array([1.]),bounds=[(0.0, None)])
            sjdf['opt_betastar'] = result.x[0]
            sjdf['opt_loss'] = result.fun
            sjdf['opt_BIC'] = utils.compute_BIC(1, Ntrials, result.fun)
            sjdf['opt_rel_entropy'] = utils.compute_rel_entropy(sj_choice_probs, fully_opt_model(result.x))
            if sjdf['opt_BIC'].iloc[0] < sjdf['best_model_BIC'].iloc[0]:
                sjdf['best_model'] = 'fully_optimal'
                sjdf['best_model_betastar'] = result.x[0]
                sjdf['best_model_loss'] = result.fun
                sjdf['best_model_BIC'] = sjdf['opt_BIC'].iloc[0]
                sjdf['best_model_rel_entropy'] = sjdf['opt_rel_entropy'].iloc[0]

            fullparam_model = lambda params: fullparam_model_horses(params,shapes_emp_,'midWS')
            fullparam_obj = lambda params: utils.compute_neg_sum_log_likelihood(choices,fullparam_model,params)
            result = minimize(fullparam_obj,np.array([1., 1.]),bounds=[(0.0, None),(0.0, None)])
            sjdf['fullparam_betastar'] = result.x[0]
            sjdf['fullparam_WSratio'] = result.x[1]
            sjdf['fullparam_loss'] = result.fun
            sjdf['fullparam_BIC'] = utils.compute_BIC(2, Ntrials, result.fun)
            sjdf['fullparam_rel_entropy'] = utils.compute_rel_entropy(sj_choice_probs, fullparam_model(result.x))
            if sjdf['fullparam_BIC'].iloc[0] < sjdf['best_model_BIC'].iloc[0]:
                sjdf['best_model'] = 'fullparam'
                sjdf['best_model_betastar'] = result.x[0]
                sjdf['best_model_loss'] = result.fun
                sjdf['best_model_BIC'] = sjdf['fullparam_BIC'].iloc[0]
                sjdf['best_model_rel_entropy'] = sjdf['fullparam_rel_entropy'].iloc[0]

            modeldf_dict[exp] = pd.concat((modeldf_dict[exp], sjdf), ignore_index=True)

    print(f'Experiment {exp} done')

if save_data_flag:
    modeldf_dict['lowWS'].to_csv(horses_datadir + 'lowWS/model_fits.csv', index=False)
    modeldf_dict['midWS'].to_csv(horses_datadir + 'midWS_learning/midWS_model_fits.csv', index=False)
    modeldf_dict['highWS'].to_csv(horses_datadir + 'highWS/model_fits.csv', index=False)
    modeldf_dict['learning'].to_csv(horses_datadir + 'midWS_learning/learning_model_fits.csv', index=False)
    modeldf_dict['speed_accuracy'].to_csv(horses_datadir + 'speed_accuracy/model_fits.csv', index=False)
    modeldf_dict['reward'].to_csv(horses_datadir + 'reward/model_fits.csv', index=False)

/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: divide by zero encountered in log
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: invalid value encountered in multiply
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: divide by zero encountered in log
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: invalid value encountered in multiply
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: divide by zero encountered in log
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_pr

Experiment lowWS done


/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: divide by zero encountered in log
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: invalid value encountered in multiply
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: divide by zero encountered in log
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: invalid value encountered in multiply
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: divide by zero encountered in log
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_pr

Experiment midWS done


/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: divide by zero encountered in log
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: invalid value encountered in multiply
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: divide by zero encountered in log
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: invalid value encountered in multiply
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: divide by zero encountered in log
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_pr

...

/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: divide by zero encountered in log
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: invalid value encountered in multiply
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: divide by zero encountered in log
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: invalid value encountered in multiply
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: divide by zero encountered in log
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_pr

Experiment reward done


/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: divide by zero encountered in log
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: invalid value encountered in multiply
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: divide by zero encountered in log
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: invalid value encountered in multiply
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: divide by zero encountered in log
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_pr

Experiment speed_accuracy done


/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: divide by zero encountered in log
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: invalid value encountered in multiply
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: divide by zero encountered in log
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: invalid value encountered in multiply
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_probs)) / sj_choice_probs.shape[0]
/workspaces/human-inference-IB/utilities.py:708: RuntimeWarning: divide by zero encountered in log
  return np.nansum(sj_choice_probs*np.log(sj_choice_probs/model_choice_pr